
# Notebook 2 - Final RAG Pipeline with HF LLM Query Rewriting + Reranking

This is the final Notebook 2 for your project.

Locked architecture:
1. Coordinator Agent
2. Retrieval Agent
3. Address Standardization Agent
4. Commute & Confidence Agent
5. Eligibility Agent
6. Escalation Agent

This notebook builds only the Retrieval Agent.

What it does:
- installs `uv`
- installs project packages with `uv pip`
- reads KB and employee data from BigQuery
- converts KB rows into LangChain Documents
- creates embeddings with Hugging Face
- builds a FAISS vector store
- uses a Hugging Face instruct LLM for query rewriting
- reranks retrieved documents
- supports batch retrieval


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/address-intelligence-project"
os.makedirs(project_path, exist_ok=True)

folders = ["notebooks", "src", "data", "scripts", "docs"]

for f in folders:
    os.makedirs(os.path.join(project_path, f), exist_ok=True)


## Step 0: install uv

We use uv first because it handles package installs more cleanly than plain pip.
If your runtime is fresh, run this cell once.


In [ ]:

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"
!uv --version



## Step 1: install project packages with uv

If Colab asks, restart runtime after this cell, then run the notebook from the top again.


In [ ]:

import os

os.environ["PATH"] += ":/root/.local/bin"

#!uv pip install --system   numpy==1.26.4   langchain   langchain-community   langchain-huggingface   faiss-cpu   sentence-transformers   transformers   accelerate   bitsandbytes   google-cloud-bigquery   db-dtypes   pandas   requests==2.32.4


In [ ]:
!pip uninstall -y numpy pandas faiss-cpu sentence-transformers transformers accelerate bitsandbytes langchain langchain-core langchain-community langchain-huggingface
!pip install --no-cache-dir \
  numpy==1.26.4 \
  pandas==2.2.2 \
  langchain \
  langchain-community \
  langchain-huggingface \
  faiss-cpu \
  sentence-transformers \
  transformers \
  accelerate \
  bitsandbytes \
  google-cloud-bigquery \
  db-dtypes \
  requests==2.32.4

In [ ]:
import numpy as np
import pandas as pd
import torch

print(np.__version__)
print(pd.__version__)
print(torch.cuda.is_available())

In [ ]:

import json
import torch
import pandas as pd
from google.colab import auth
from google.cloud import bigquery
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig



## Step 2: authenticate and connect to BigQuery


In [ ]:

auth.authenticate_user()
print("Authenticated successfully")

PROJECT_ID = "" #add your project id here
DATASET_ID = "address_intelligence_demo" #add your database id here
TABLE_EMPLOYEES = "employee_input"
TABLE_KB = "kb_documents"

client = bigquery.Client(project=PROJECT_ID)
print("Connected to BigQuery project:", PROJECT_ID)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))



## Step 3: read employee and KB data from BigQuery


In [ ]:

kb_query = f'''
SELECT doc_id, source_type, title, text
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_KB}`
ORDER BY doc_id
'''

employee_query = f'''
SELECT employee_id, employee_name, office_id, raw_home_address, office_address
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_EMPLOYEES}`
ORDER BY employee_id
'''

kb_df = client.query(kb_query).to_dataframe()
employee_df = client.query(employee_query).to_dataframe()

print("KB rows:", len(kb_df))
print("Employee rows:", len(employee_df))
kb_df.head(10)



## Step 4: convert KB rows into LangChain Documents


In [ ]:

documents = []
for _, row in kb_df.iterrows():
    documents.append(
        Document(
            page_content=row["text"],
            metadata={
                "doc_id": row["doc_id"],
                "source_type": row["source_type"],
                "title": row["title"],
            }
        )
    )

print("Created LangChain documents:", len(documents))
documents[0]



## Step 5: create embeddings and vector store


In [ ]:

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

VECTOR_DIR = "rag_vectorstore_final"
vectorstore.save_local(VECTOR_DIR)

print("Embedding model ready:", EMBEDDING_MODEL_NAME)
print("FAISS vector store created")
print("Saved vector store to:", VECTOR_DIR)



## Step 6: load a Hugging Face instruct LLM for query rewriting

Recommended model:
- Qwen/Qwen2.5-3B-Instruct

If needed, you can later swap to:
- microsoft/Phi-3.5-mini-instruct


In [ ]:

REWRITE_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = AutoTokenizer.from_pretrained(REWRITE_MODEL_NAME)

rewrite_model = AutoModelForCausalLM.from_pretrained(
    REWRITE_MODEL_NAME,
    quantization_config=bnb_config if torch.cuda.is_available() else None,
    device_map="auto",
    trust_remote_code=True
)

rewrite_pipe = pipeline(
    "text-generation",
    model=rewrite_model,
    tokenizer=tokenizer,
    max_new_tokens=160,
    temperature=0.0,
    do_sample=False
)

print("LLM query rewriter ready:", REWRITE_MODEL_NAME)



## Step 7: create system prompt + user prompt for LLM query rewriting


In [ ]:

SYSTEM_PROMPT_QUERY_REWRITER = """
You are a retrieval query rewriting assistant for an enterprise address intelligence system.

Your job:
1. Read a messy employee-entered home address and an assigned office address.
2. Rewrite them into a clean, retrieval-friendly query.
3. Preserve important location details.
4. Expand the retrieval intent so the retriever can find:
   - commute eligibility policy
   - address standardization rules
   - typo correction examples
   - office metadata
   - manual review / escalation rules
5. Do not invent unsupported address fields.
6. Return only the rewritten retrieval query text.
"""

def build_user_prompt_for_rewriter(raw_home_address: str, office_address: str) -> str:
    return f"""
Raw employee home address:
{raw_home_address}

Assigned office address:
{office_address}

Rewrite this into a retrieval query that will help search the enterprise KB for the best supporting documents.
"""



## Step 8: build the HF LLM query rewriting function


In [ ]:

def rewrite_query_with_hf_llm(raw_home_address: str, office_address: str, text_generation_pipeline) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT_QUERY_REWRITER},
        {"role": "user", "content": build_user_prompt_for_rewriter(raw_home_address, office_address)},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    output = text_generation_pipeline(prompt)[0]["generated_text"]
    rewritten = output[len(prompt):].strip()
    return rewritten



## Step 9: test LLM query rewriting on one employee row


In [ ]:

sample_row = employee_df.iloc[2]
sample_row


In [ ]:

raw_address = sample_row["raw_home_address"]
office_address = sample_row["office_address"]

rewritten_query = rewrite_query_with_hf_llm(
    raw_home_address=raw_address,
    office_address=office_address,
    text_generation_pipeline=rewrite_pipe
)

print("RAW ADDRESS:")
print(raw_address)
print("\nREWRITTEN QUERY:")
print(rewritten_query)



## Step 10: retrieve candidate docs from the vector store


In [ ]:

initial_docs = vectorstore.similarity_search(rewritten_query, k=6)

for i, doc in enumerate(initial_docs, start=1):
    print("=" * 80)
    print("Initial Rank:", i)
    print("Doc ID:", doc.metadata["doc_id"])
    print("Title:", doc.metadata["title"])
    print("Source type:", doc.metadata["source_type"])
    print("Text:", doc.page_content)



## Step 11: rerank the candidate docs


In [ ]:

RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)

print("Reranker ready:", RERANKER_MODEL_NAME)


In [ ]:

def rerank_documents(query_text, docs, reranker_model, top_k=4):
    pairs = [[query_text, d.page_content] for d in docs]
    scores = reranker_model.predict(pairs)

    scored = list(zip(docs, scores))
    scored = sorted(scored, key=lambda x: x[1], reverse=True)

    final_docs = []
    for doc, score in scored[:top_k]:
        doc.metadata["rerank_score"] = float(score)
        final_docs.append(doc)
    return final_docs

reranked_docs = rerank_documents(
    query_text=rewritten_query,
    docs=initial_docs,
    reranker_model=reranker,
    top_k=4
)

for i, doc in enumerate(reranked_docs, start=1):
    print("=" * 80)
    print("Final Rank:", i)
    print("Doc ID:", doc.metadata["doc_id"])
    print("Title:", doc.metadata["title"])
    print("Rerank score:", round(doc.metadata["rerank_score"], 4))
    print("Text:", doc.page_content)



## Step 12: package everything into the final Retrieval Agent function


In [ ]:

def retrieval_agent_final(
    raw_home_address,
    office_address,
    vectorstore,
    text_generation_pipeline,
    reranker_model,
    candidate_k=8,
    top_k=4
):
    rewritten_query = rewrite_query_with_hf_llm(
        raw_home_address=raw_home_address,
        office_address=office_address,
        text_generation_pipeline=text_generation_pipeline
    )

    candidate_docs = vectorstore.similarity_search(rewritten_query, k=candidate_k)
    final_docs = rerank_documents(rewritten_query, candidate_docs, reranker_model, top_k=top_k)

    return {
        "raw_home_address": raw_home_address,
        "office_address": office_address,
        "rewritten_query": rewritten_query,
        "retrieved_context": [
            {
                "doc_id": d.metadata["doc_id"],
                "title": d.metadata["title"],
                "source_type": d.metadata["source_type"],
                "rerank_score": d.metadata.get("rerank_score"),
                "text": d.page_content
            }
            for d in final_docs
        ]
    }

retrieval_output = retrieval_agent_final(
    raw_home_address=raw_address,
    office_address=office_address,
    vectorstore=vectorstore,
    text_generation_pipeline=rewrite_pipe,
    reranker_model=reranker,
    candidate_k=8,
    top_k=4
)

retrieval_output



## Step 13: batch retrieval for many employees


In [ ]:

batch_outputs = []

for _, row in employee_df.head(5).iterrows():
    result = retrieval_agent_final(
        raw_home_address=row["raw_home_address"],
        office_address=row["office_address"],
        vectorstore=vectorstore,
        text_generation_pipeline=rewrite_pipe,
        reranker_model=reranker,
        candidate_k=8,
        top_k=4
    )

    batch_outputs.append({
        "employee_id": row["employee_id"],
        "employee_name": row["employee_name"],
        "raw_home_address": row["raw_home_address"],
        "rewritten_query": result["rewritten_query"],
        "top_doc_titles": [x["title"] for x in result["retrieved_context"]]
    })

batch_df = pd.DataFrame(batch_outputs)
batch_df


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 14: save retrieval artifacts locally for later notebooks


In [ ]:

with open("retrieval_final_sample_output.json", "w") as f:
    json.dump(retrieval_output, f, indent=2)

batch_df.to_csv("retrieval_final_batch_preview.csv", index=False)

print("Saved files:")
print("- retrieval_final_sample_output.json")
print("- retrieval_final_batch_preview.csv")
print("- rag_vectorstore_final/")



## Finished

This file includes:
- BigQuery KB loading
- LangChain Documents
- Hugging Face embeddings
- FAISS vector search
- HF LLM-based query rewriting with system + user prompts
- reranking
- batch retrieval


Creating helper functions (optional)

In [ ]:


from google.colab import drive
drive.mount('/content/drive')

import os

project_path = "/content/drive/MyDrive/address-intelligence-project"
os.listdir("/content/drive/MyDrive/address-intelligence-project")
os.listdir("/content/drive/MyDrive/address-intelligence-project/src")

In [ ]:
%%writefile /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py

def test_func():
    return "hello"

In [ ]:
os.listdir("/content/drive/MyDrive/address-intelligence-project/src")

In [ ]:
%%writefile /content/drive/MyDrive/address-intelligence-project/src/retrieval_helper.py

from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

def init_retrieval_components(documents):
    embeddings = HuggingFaceEmbeddings()
    vectorstore = FAISS.from_documents(documents, embeddings)
    return vectorstore

def retrieval_agent(query, vectorstore, k=3):
    return vectorstore.similarity_search(query, k=k)